In [10]:
# 1. SETUP & IMPORTS
# ==========================================
import pandas as pd
import numpy as np
from scipy.signal import savgol_filter, find_peaks
from scipy.fftpack import dct
from scipy.stats import skew, kurtosis, entropy
import matplotlib.pyplot as plt
import os


In [11]:
# --- CONFIGURATION ---
# Paths for local workspace
import os

# Get the project root directory (parent of notebooks folder)
# In Jupyter notebooks, we need to use os.getcwd() and navigate up
BASE_PATH = os.path.dirname(os.getcwd())

# UPDATED: Use synthetic augmented dataset (1035 samples instead of 35)
INPUT_FILE = os.path.join(BASE_PATH, 'datasets', 'synthetic_augmented_data.csv')

OUTPUT_FEATURES = os.path.join(BASE_PATH, 'datasets', 'X_features.csv')
OUTPUT_TARGETS = os.path.join(BASE_PATH, 'datasets', 'y_targets.csv')

print(f"Base path: {BASE_PATH}")
print(f"Input file: {INPUT_FILE}")
print("🚀 Using SYNTHETIC AUGMENTED DATASET (1035 samples)")


Base path: /Users/abhijitchaudhuri/Downloads/Orange_freshness_project
Input file: /Users/abhijitchaudhuri/Downloads/Orange_freshness_project/datasets/synthetic_augmented_data.csv
🚀 Using SYNTHETIC AUGMENTED DATASET (1035 samples)


In [12]:
# 2. HELPER FUNCTIONS (The Math)
# ==========================================

def apply_savgol(signal, window=11, poly=3):
    """
    Applies Savitzky-Golay smoothing.
    Analogy: The 'Smart Iron' that smooths wrinkles but keeps the shape.
    """
    return savgol_filter(signal, window_length=window, polyorder=poly)

def extract_chemometric_features(signal, voltage_axis):
    """
    Extracts domain-specific features.
    Focus: The specific peak at 0.85V which correlates to Folic Acid.
    """
    # Find the index closest to 0.85V
    target_voltage = 0.85
    idx = (np.abs(voltage_axis - target_voltage)).argmin()

    peak_height_085 = signal[idx]
    return peak_height_085

def extract_statistical_features(signal):
    """
    Extracts general shape features.
    """
    return {
        'mean': np.mean(signal),
        'std_dev': np.std(signal),
        'energy': np.sum(signal**2), # Total energy in signal
        'skewness': skew(signal),    # Is the curve leaning left or right?
        'kurtosis': kurtosis(signal) # How sharp is the peak?
    }


In [13]:
# 3. MAIN PIPELINE
# ==========================================

import glob

# A. Load Synthetic Augmented Data
print("="*60)
print("LOADING SYNTHETIC AUGMENTED DATA")
print("="*60)
try:
    df_all = pd.read_csv(INPUT_FILE)
    print(f"Data loaded: {df_all.shape}")
    print(f"Total samples: {len(df_all)}")
    
    # Attach folic acid ground-truth if available
    key_path = os.path.join(BASE_PATH, 'datasets', 'key.csv')
    if os.path.exists(key_path):
        key_df = pd.read_csv(key_path)
        if 'true_conc_uM' in key_df.columns:
            df_all = df_all.merge(key_df[['filename', 'true_conc_uM']], on='filename', how='left')
            labeled = df_all['true_conc_uM'].notna().sum()
            print(f"\nFolic acid labels attached for {labeled} samples")
        else:
            print("\nWarning: key.csv found but missing 'true_conc_uM' column")
    else:
        print("\nWarning: key.csv not found; folic acid labels unavailable")
    
    # Show batch distribution
    batch_counts = df_all['batch'].value_counts().sort_index()
    print(f"\nBatch Distribution:")
    for batch, count in batch_counts.items():
        print(f"  Batch {batch}: {count} samples")
    
    # Show day range
    print(f"\nDay Range: {df_all['day'].min():.2f} to {df_all['day'].max():.2f} days")
    
    # Separate real vs synthetic for info
    real_samples = (df_all['batch'] != 'Synthetic').sum()
    synthetic_samples = (df_all['batch'] == 'Synthetic').sum()
    print(f"\nComposition:")
    print(f"  Real samples: {real_samples}")
    print(f"  Synthetic samples: {synthetic_samples}")
    
except FileNotFoundError:
    print(f"ERROR: File not found at {INPUT_FILE}")
    print("Please run 'generate_data.py' first to create synthetic data.")
    raise

# B. Load Test Data (Batches 6-7) from test_dataset_blind folder
# This is kept for validation on real unseen data
print("\n" + "="*60)
print("LOADING REAL TEST DATA (Batches 6-7 from test_dataset_blind/)")
print("="*60)

test_folder = os.path.join(BASE_PATH, 'datasets', 'test_dataset_blind')
test_files = sorted(glob.glob(os.path.join(test_folder, '*.csv')))
print(f"Found {len(test_files)} real test files")

if len(test_files) > 0:
    test_data_list = []
    for test_file in test_files:
        # Parse batch and day from filename (e.g., batch6_day1.csv)
        filename = os.path.basename(test_file)
        parts = filename.replace('.csv', '').split('_')
        batch = int(parts[0].replace('batch', ''))
        day = int(parts[1].replace('day', ''))
        
        # Load the raw sensor data
        df_raw = pd.read_csv(test_file)
        # This file has: voltage, current_1, current_2, ..., current_15
        # We need to transpose and aggregate to match training data structure
        
        # Take mean across all current columns for each voltage point
        current_cols = [col for col in df_raw.columns if col.startswith('current_')]
        sensor_values = df_raw[current_cols].mean(axis=1).values
        
        # Create a row with metadata + sensor features
        row_dict = {'batch': batch, 'day': day, 'filename': filename}
        for idx, val in enumerate(sensor_values, 1):
            row_dict[f'f{idx}'] = val
        
        test_data_list.append(row_dict)
    
    df_test_real = pd.DataFrame(test_data_list)
    print(f"Real test data processed: {df_test_real.shape}")
    print(f"Batches in real test: {sorted(df_test_real['batch'].unique())}")
    print(f"Days in real test: {sorted(df_test_real['day'].unique())}")
    
    # C. Combine ALL Data (Synthetic + Real Training + Real Test)
    print("\n" + "="*60)
    print("COMBINING ALL DATA (Synthetic + Real Test)")
    print("="*60)
    
    # Ensure both dataframes have the same columns (including true_conc_uM if present)
    all_columns = sorted(set(df_all.columns) | set(df_test_real.columns))
    df_all = df_all.reindex(columns=all_columns, fill_value=0)
    df_test_real = df_test_real.reindex(columns=all_columns, fill_value=np.nan)
    
    df_all = pd.concat([df_all, df_test_real], ignore_index=True)
    print(f"Final combined data shape: {df_all.shape}")
    print(f"Total samples: {len(df_all)}")
else:
    print("No real test files found - using synthetic data only")

# D. Extract Features from Combined Data
print("\n" + "="*60)
print("FEATURE EXTRACTION")
print("="*60)

# Get sensor feature columns
feature_cols = [col for col in df_all.columns if col.startswith('f') and col != 'filename']
X_raw = df_all[feature_cols].values.astype(np.float64)
y = df_all[['batch', 'day', 'true_conc_uM']].copy()

# Create voltage axis (assuming linear sweep from -0.2V to 1.0V)
n_features = len(feature_cols)
voltage_axis = np.linspace(-0.2, 1.0, n_features)

# Initialize storage
X_features = []

print(f"Processing {len(X_raw)} samples...")
print(f"Each sample has {n_features} raw sensor points")

for i, raw_signal in enumerate(X_raw):
    # Step 1: Smooth the signal
    smoothed = apply_savgol(raw_signal)
    
    # Step 2: Extract chemometric feature (Peak at 0.85V)
    peak_085V = extract_chemometric_features(smoothed, voltage_axis)
    
    # Step 3: Statistical features
    mean = np.mean(smoothed)
    std = np.std(smoothed)
    energy = np.sum(smoothed**2)  # Total signal energy
    skewness = skew(smoothed)
    kurt = kurtosis(smoothed)
    
    # Step 4: Discrete Cosine Transform (Frequency domain features)
    dct_coeffs = dct(smoothed, norm='ortho')
    dct_features = dct_coeffs[1:6]  # Take first 5 DCT coefficients (ignore DC)
    
    # Combine all features into a single row
    feature_row = [
        peak_085V,    # Chemometric
        mean, std, energy,  # Statistical
        skewness, kurt,     # Shape
        *dct_features       # Frequency (5 features)
    ]
    
    X_features.append(feature_row)

# Convert to DataFrame
feature_names = [
    'Peak_0.85V', 
    'Mean', 'Std_Dev', 'Energy',
    'Skewness', 'Kurtosis',
    'DCT_1', 'DCT_2', 'DCT_3', 'DCT_4', 'DCT_5'
]

X_df = pd.DataFrame(X_features, columns=feature_names)

print("\n" + "="*60)
print("FEATURE EXTRACTION COMPLETE")
print("="*60)
print(f"Final feature matrix shape: {X_df.shape}")
print(f"Features: {list(X_df.columns)}")
print(f"\nFeature Summary:")
print(X_df.describe())

# E. Save Processed Data
print("\n" + "="*60)
print("SAVING PROCESSED DATA")
print("="*60)

X_df.to_csv(OUTPUT_FEATURES, index=False)
y.to_csv(OUTPUT_TARGETS, index=False)

print(f"✅ Features saved to: {OUTPUT_FEATURES}")
print(f"✅ Targets saved to: {OUTPUT_TARGETS}")

print("\n" + "="*60)
print("PREPROCESSING COMPLETE - Ready for Modeling!")
print("="*60)
print(f"Total samples ready for training: {len(X_df)}")
# Handle mixed types in batch column (int and 'Synthetic' string)
unique_batches = y['batch'].unique()
print(f"Batches: {list(unique_batches)[:10]}...")  # Show first 10 to avoid sorting mixed types


LOADING SYNTHETIC AUGMENTED DATA
Data loaded: (2035, 3708)
Total samples: 2035

Folic acid labels attached for 35 samples

Batch Distribution:
  Batch 1: 207 samples
  Batch 2: 207 samples
  Batch 3: 207 samples
  Batch 4: 207 samples
  Batch 5: 207 samples
  Batch 6: 200 samples
  Batch 7: 200 samples
  Batch 8: 200 samples
  Batch 9: 200 samples
  Batch 10: 200 samples

Day Range: 0.00 to 14.00 days

Composition:
  Real samples: 2035
  Synthetic samples: 0

LOADING REAL TEST DATA (Batches 6-7 from test_dataset_blind/)
Found 15 real test files
Real test data processed: (15, 250)
Batches in real test: [np.int64(6), np.int64(7)]
Days in real test: [np.int64(0), np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5), np.int64(6), np.int64(7), np.int64(8), np.int64(9), np.int64(10), np.int64(11), np.int64(12), np.int64(13), np.int64(14)]

COMBINING ALL DATA (Synthetic + Real Test)
Final combined data shape: (2050, 3709)
Total samples: 2050

FEATURE EXTRACTION
Processing 2050 samp